# Phase 1 — TF-IDF Pure Text Representation Experiment (Group K-Fold Cross-Validation)

Eksperimen representasi teks berbasis TF-IDF (Murni Teks / PureText tanpa fitur numerik bisnis) untuk prioritasi kebutuhan perangkat lunak (Functional Requirements dan Non-Functional Requirements). Notebook ini mencakup ekstraksi matriks TF-IDF (100 fitur unigram/bigram terpilih), integrasi tipe kebutuhan (FR/NFR), pembentukan query group per proyek, **pembagian data berbasis Group K-Fold Cross-Validation (project_id)**, pelatihan model ranking LightGBM LambdaRank per fold, evaluasi metrik (NDCG@5, NDCG@10, MAP, Spearman, Kendall Tau) dengan agregasi Mean ± Std Dev, serta pencatatan artefak dan metrik ke MLflow server.

In [1]:
%pip install mlflow lightgbm joblib scikit-learn

import logging
import os
import sys
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')
    sys.stderr.reconfigure(encoding='utf-8')
import json
import joblib
import warnings
import itertools
import requests

try:
    from IPython.display import display
except ImportError:
    display = print

import mlflow
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.feature_selection import SelectKBest, chi2, mutual_info_classif
from sklearn.metrics import ndcg_score, average_precision_score
from scipy.stats import spearmanr, kendalltau
import lightgbm as lgb

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
MLFLOW_URI = "https://mlflow.smbgarasibmw.my.id"

# Koneksi ke tracking server MLflow dengan fallback lokal
server_online = False
try:
    resp = requests.get(MLFLOW_URI, timeout=10)
    if resp.status_code == 200:
        mlflow.set_tracking_uri(MLFLOW_URI)
        mlflow.set_experiment("Phase_1_TFIDF_Text_Representation")
        logger.info(f"Successfully connected to MLflow tracking server: {MLFLOW_URI}")
        server_online = True
    else:
        logger.warning(f"MLflow server at {MLFLOW_URI} returned HTTP {resp.status_code}. Using local MLflow store (./mlruns).")
except Exception as e:
    logger.warning(f"MLflow server unreachable ({e}). Using local MLflow store (./mlruns).")

if not server_online:
    mlflow.set_tracking_uri("file:./mlruns")
    mlflow.set_experiment("Phase_1_TFIDF_Text_Representation")

Cell 1 executed successfully.


## Konfigurasi Parameter Eksperimen

Pengaturan path dataset TF-IDF PureText, direktori keluaran, seed acak, metode seleksi fitur teks, serta jumlah lipatan Group K-Fold Cross-Validation.

In [2]:
# Path dataset PureText (otomatis mencari di direktori lokal riset-phase-1/dataset atau output_datasets)
DATASET_PATH = "dataset/Dataset_Training_TFIDF_PureText.csv"
if not os.path.exists(DATASET_PATH):
    DATASET_PATH = "../output_datasets/Dataset_Training_TFIDF_PureText.csv"

OUTPUT_DIR = "outputs/phase1_tfidf_gkf"
RANDOM_STATE = 42
FEATURE_SELECTION_METHOD = "none"  # Fitur teks sudah terseleksi 100 fitur terbaik ('none' / 'chi_square')
K_FEATURES = 100
N_SPLITS = 5

os.makedirs(OUTPUT_DIR, exist_ok=True)
logger.info(f"Using Dataset: {DATASET_PATH}")
logger.info(f"Output directory initialized at: {OUTPUT_DIR}")

Cell 3 executed successfully.


## STEP 1 — Pemuatan Dataset dan Validasi Skema PureText

Memuat dataset TF-IDF PureText, memeriksa dimensi matriks, keberadaan kolom wajib untuk ranking (`project_id`, `rank`, `type_encoded`), serta mengurutkan data berdasarkan `project_id`.

In [3]:
logger.info(f"Loading dataset from {DATASET_PATH}...")
df = pd.read_csv(DATASET_PATH)

logger.info(f"Dataset shape: {df.shape}")
logger.info(f"Columns count: {len(df.columns)}")

# Validasi kolom wajib untuk ranking PureText
required_cols = ["project_id", "rank", "type_encoded"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Required columns missing: {missing}")

logger.info("All required columns present for PureText TF-IDF ranking pipeline.")

# Urutkan berdasarkan project_id agar data per proyek terkelompok rapi
df = df.sort_values(by="project_id").reset_index(drop=True)
logger.info("Dataset sorted by 'project_id' for ranking context.")
display(df.head(3))

Cell 5 executed successfully.


## STEP 2 — Validasi Kualitas Data

Pemeriksaan kualitas data yang meliputi identifikasi nilai hilang (missing values), duplikasi baris, keabsahan project_id dan nilai rank, serta pengecekan tipe data seluruh kolom.

In [4]:
validation_report = {}

# Cek missing values
missing_counts = df.isnull().sum()
missing_cols = missing_counts[missing_counts > 0]
validation_report["missing_values"] = len(missing_cols)
if len(missing_cols) > 0:
    logger.warning(f"Columns with missing values: {missing_cols.to_dict()}")
else:
    logger.info("No missing values found.")

# Cek duplikasi data
dup_rows = df.duplicated().sum()
validation_report["duplicate_rows"] = int(dup_rows)
if dup_rows > 0:
    logger.warning(f"Duplicate rows: {dup_rows}")
else:
    logger.info("No duplicate rows found.")

# Validasi project_id dan rank
invalid_pid = df["project_id"].isnull().sum() + (df["project_id"].astype(str).str.strip() == "").sum()
validation_report["invalid_project_ids"] = int(invalid_pid)
if invalid_pid > 0:
    logger.warning(f"Invalid project IDs: {invalid_pid}")
else:
    logger.info("All project IDs are valid.")

invalid_rank = (~pd.to_numeric(df["rank"], errors="coerce").notna()).sum()
validation_report["invalid_rank"] = int(invalid_rank)
if invalid_rank > 0:
    logger.warning(f"Invalid rank values: {invalid_rank}")
else:
    logger.info("All rank values are valid.")

validation_report["dtypes"] = {c: str(dt) for c, dt in df.dtypes.items()}

logger.info("=== Validation Report ===")
for k, v in validation_report.items():
    if k != "dtypes":
        logger.info(f"  {k}: {v}")

Cell 7 executed successfully.


## STEP 3 — Ekstraksi Matriks Fitur TF-IDF

Mengambil seluruh kolom fitur representasi teks TF-IDF yang diawali dengan `tfidf_`.

In [5]:
# Ambil kolom representasi TF-IDF
tfidf_cols = [c for c in df.columns if c.startswith("tfidf_")]
X_tfidf = df[tfidf_cols].values

logger.info(f"Extracted TF-IDF matrix with {X_tfidf.shape[1]} features.")
logger.info(f"TF-IDF matrix shape: {X_tfidf.shape}")

Cell 9 executed successfully.


## STEP 4 — Penanganan Dimensi Fitur Teks TF-IDF

Pada dataset PureText, 100 fitur teks TF-IDF unigram & bigram telah melalui proses kurasi dan seleksi berbasis Chi-Square pada tahap feature engineering. Seluruh 100 fitur teks ini dipertahankan secara utuh.

In [6]:
if FEATURE_SELECTION_METHOD == "chi_square" and K_FEATURES < X_tfidf.shape[1]:
    y_rank = df["rank"].values
    selector = SelectKBest(score_func=chi2, k=K_FEATURES)
    X_text_selected = selector.fit_transform(X_tfidf, y_rank)
    selected_indices = selector.get_support(indices=True)
    selected_tfidf_cols = [tfidf_cols[i] for i in selected_indices]
    logger.info(f"Reduced TF-IDF text matrix shape: {X_text_selected.shape}")
else:
    X_text_selected = X_tfidf
    selected_tfidf_cols = tfidf_cols
    logger.info(f"Retained all pre-selected TF-IDF features: {len(selected_tfidf_cols)} features.")

logger.info(f"Text feature matrix shape: {X_text_selected.shape}")

Cell 11 executed successfully.


## STEP 5 — Representasi Teks Murni (PureText — Tanpa Fitur Numerik Bisnis)

Sesuai tujuan penelitian Phase 1 (Pure Text Representation), eksperimen ini **tidak menyertakan fitur numerik bisnis** (seperti value, effort, risk, stakeholder_priority) guna menguji kapabilitas murni representasi teks dalam prioritasi kebutuhan perangkat lunak.

In [7]:
logger.info("Phase 1 Experiment: Pure Text Representation (No business numeric features).")
X_text_dense = X_text_selected.toarray() if hasattr(X_text_selected, "toarray") else X_text_selected
fused_cols = selected_tfidf_cols

logger.info(f"Pure text representation dimension: {X_text_dense.shape[1]}")

Cell 13 executed successfully.


## STEP 6 — Integrasi Fitur Tipe Kebutuhan dan Pembentukan Matriks Fitur Akhir X

Mengintegrasikan fitur kategori `type_encoded` (Functional Requirement = 1, Non-Functional Requirement = 0) ke dalam matriks fitur akhir X.

In [8]:
logger.info("Integrating 'type_encoded' feature into final matrix X...")

if "type_encoded" in df.columns:
    X_type = df[["type_encoded"]].values
    X_matrix = np.hstack((X_text_dense, X_type))
    all_feature_cols = fused_cols + ["type_encoded"]
else:
    X_matrix = X_text_dense
    all_feature_cols = fused_cols

X = pd.DataFrame(X_matrix, columns=all_feature_cols)

logger.info("Feature integration completed.")
logger.info(f"  Final DataFrame X shape: {X.shape}")

Cell 15 executed successfully.


## STEP 7 — Pembentukan Query Group dan Label Relevansi

Mengonversi kolom rank menjadi label relevansi terurut per proyek (0 = prioritas terendah, N-1 = prioritas tertinggi) untuk objektivitas LightGBM LambdaRank dan mendefinisikan kelompok proyek (groups) berdasarkan `project_id`.

In [9]:
logger.info("Transforming ranks into LambdaRank relevance labels...")

# Transformasi rank ke label relevansi per proyek (0 = terendah, nilai lebih tinggi = lebih relevan)
df["label"] = df.groupby("project_id")["rank"].transform(
    lambda x: x.rank(method="dense", ascending=False).astype(int) - 1
)

y = df["label"].values
groups = df["project_id"].values

group_sizes = pd.Series(groups).value_counts()
num_groups = len(group_sizes)

logger.info(f"Number of query groups (projects): {num_groups}")
logger.info(f"Group size stats:\n{group_sizes.describe()}")

logger.info(f"Label validation -> Range: [{y.min()}, {y.max()}], Unique labels count: {len(np.unique(y))}")

Cell 17 executed successfully.


## STEP 8 — Inisialisasi Group K-Fold Cross-Validation

Membagi dataset menggunakan GroupKFold(n_splits=N_SPLITS) berdasarkan `project_id`. Pendekatan ini menjamin bahwa setiap lipatan (fold) menguji model pada proyek-proyek yang sama sekali belum pernah dilihat pada data latih (zero project overlap), sehingga mengeliminasi data leakage.

In [10]:
gkf = GroupKFold(n_splits=N_SPLITS)

logger.info(f"Initialized GroupKFold with {N_SPLITS} splits on project_id.")
for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups=groups), 1):
    train_projs = set(groups[train_idx])
    val_projs = set(groups[val_idx])
    overlap = train_projs & val_projs
    logger.info(f"Fold {fold}: Train projects={len(train_projs)}, Val projects={len(val_projs)}, Overlap={len(overlap)}")
    if len(overlap) > 0:
        raise ValueError(f"Data leakage detected in Fold {fold}: overlap={overlap}")

logger.info("Zero project overlap verified across all folds!")

Cell 19 executed successfully.


## STEP 9 — Pelatihan dan Evaluasi LightGBM LambdaRank per Fold (Cross-Validation)

Melatih model LightGBM Ranker dengan objektivitas lambdarank pada data latih setiap fold dan mengevaluasi metrik ranking (NDCG@5, NDCG@10, MAP, Spearman, Kendall Tau) pada proyek uji yang belum pernah dilihat model.

In [11]:
fold_metrics_list = []
oof_predictions = np.zeros(len(df))
models = []

logger.info(f"Starting {N_SPLITS}-Fold Group Cross-Validation...")

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups=groups), 1):
    logger.info(f"\n{'='*20} Fold {fold}/{N_SPLITS} {'='*20}")
    
    # Wajib sort index berdasarkan project_id agar query group LightGBM terkelompok rapi
    train_idx = sorted(train_idx, key=lambda i: groups[i])
    val_idx = sorted(val_idx, key=lambda i: groups[i])
    
    X_train, y_train = X.iloc[train_idx], y[train_idx]
    X_val, y_val = X.iloc[val_idx], y[val_idx]
    
    train_groups = groups[train_idx]
    val_groups = groups[val_idx]
    
    # Hitung ukuran kelompok (group query counts) per proyek
    train_group_counts = [len(list(g)) for _, g in itertools.groupby(train_groups)]
    val_group_counts = [len(list(g)) for _, g in itertools.groupby(val_groups)]
    
    max_train_group = max(train_group_counts)
    num_leaves = min(max_train_group, 255)
    
    ranker = lgb.LGBMRanker(
        objective="lambdarank",
        n_estimators=100,
        learning_rate=0.05,
        num_leaves=num_leaves,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    
    ranker.fit(
        X_train, y_train,
        group=train_group_counts,
        eval_set=[(X_val, y_val)],
        eval_group=[val_group_counts],
        eval_at=[5, 10],
        callbacks=[lgb.early_stopping(stopping_rounds=15, verbose=False)]
    )
    
    models.append(ranker)
    
    # Prediksi skor ranking untuk validasi
    val_pred = ranker.predict(X_val)
    oof_predictions[val_idx] = val_pred
    
    # Hitung metrik per proyek lalu rata-ratakan untuk fold ini
    ndcg5_list, ndcg10_list, map_list = [], [], []
    spearman_list, kendall_list = [], []
    
    unique_val_projs = []
    idx_start = 0
    for g_len in val_group_counts:
        p_slice = slice(idx_start, idx_start + g_len)
        p_y = y_val[p_slice]
        p_pred = val_pred[p_slice]
        idx_start += g_len
        
        # NDCG (memerlukan 2D array jika menggunakan ndcg_score)
        if len(np.unique(p_y)) > 1:
            try:
                n5 = ndcg_score([p_y], [p_pred], k=5)
                n10 = ndcg_score([p_y], [p_pred], k=10)
                ndcg5_list.append(n5)
                ndcg10_list.append(n10)
            except Exception:
                pass
            
            # Spearman & Kendall Tau
            sp_corr, _ = spearmanr(p_y, p_pred)
            kt_corr, _ = kendalltau(p_y, p_pred)
            if not np.isnan(sp_corr):
                spearman_list.append(sp_corr)
            if not np.isnan(kt_corr):
                kendall_list.append(kt_corr)
        
        # Binarize label untuk MAP
        binary_y = (p_y >= np.median(p_y)).astype(int)
        if len(np.unique(binary_y)) > 1:
            map_score = average_precision_score(binary_y, p_pred)
            map_list.append(map_score)
    
    fold_ndcg5 = np.mean(ndcg5_list) if ndcg5_list else ndcg_score([y_val], [val_pred], k=5)
    fold_ndcg10 = np.mean(ndcg10_list) if ndcg10_list else ndcg_score([y_val], [val_pred], k=10)
    fold_map = np.mean(map_list) if map_list else 0.0
    fold_spearman = np.mean(spearman_list) if spearman_list else spearmanr(y_val, val_pred)[0]
    fold_kendall = np.mean(kendall_list) if kendall_list else kendalltau(y_val, val_pred)[0]
    
    # Hitung p-value keseluruhan fold
    sp_all, sp_pval = spearmanr(y_val, val_pred)
    kt_all, kt_pval = kendalltau(y_val, val_pred)
    
    fold_result = {
        "fold": fold,
        "train_rows": len(train_idx),
        "val_rows": len(val_idx),
        "train_projects": len(train_group_counts),
        "val_projects": len(val_group_counts),
        "NDCG@5": fold_ndcg5,
        "NDCG@10": fold_ndcg10,
        "MAP": fold_map,
        "Spearman": fold_spearman,
        "Spearman_pvalue": sp_pval,
        "KendallTau": fold_kendall,
        "KendallTau_pvalue": kt_pval
    }
    fold_metrics_list.append(fold_result)
    logger.info(f"Fold {fold} Results -> NDCG@5: {fold_ndcg5:.4f}, NDCG@10: {fold_ndcg10:.4f}, MAP: {fold_map:.4f}, Spearman: {fold_spearman:.4f}, KendallTau: {fold_kendall:.4f}")

cv_details_df = pd.DataFrame(fold_metrics_list)
display(cv_details_df)

Cell 21 executed successfully.


## STEP 10 — Agregasi Metrik Cross-Validation (Mean ± Std Dev)

Menghitung performa agregat di seluruh lipatan pengujian mencakup rata-rata dan standar deviasi dari NDCG@5, NDCG@10, MAP, Spearman, dan Kendall Tau.

In [12]:
metric_cols = ["NDCG@5", "NDCG@10", "MAP", "Spearman", "Spearman_pvalue", "KendallTau", "KendallTau_pvalue"]
cv_summary = {}

for col in metric_cols:
    cv_summary[f"{col}_mean"] = float(cv_details_df[col].mean())
    cv_summary[f"{col}_std"] = float(cv_details_df[col].std())

summary_rows = []
for col in ["NDCG@5", "NDCG@10", "MAP", "Spearman", "KendallTau"]:
    m = cv_summary[f"{col}_mean"]
    s = cv_summary[f"{col}_std"]
    summary_rows.append({
        "Metric": col,
        "Mean ± Std": f"{m:.6f} ± {s:.6f}",
        "Mean": m,
        "Std": s
    })

cv_summary_df = pd.DataFrame(summary_rows)
logger.info("=== Cross-Validation Aggregated Summary ===")
logger.info(f"\n{cv_summary_df[['Metric', 'Mean ± Std']].to_string(index=False)}")
display(cv_summary_df)

Cell 23 executed successfully.


## STEP 11 — Penyimpanan Artefak dan Output Eksperimen

Menyimpan berkas ringkasan metrik CV JSON, tabel detail fold CSV, out-of-fold predictions CSV, daftar fitur, dan model terbaik ke direktori keluaran.

In [13]:
# Simpan model terbaik (berdasarkan NDCG@5)
best_fold_idx = int(cv_details_df["NDCG@5"].argmax())
best_model = models[best_fold_idx]
best_model_path = os.path.join(OUTPUT_DIR, "lgbm_ranker_best_fold.pkl")
joblib.dump(best_model, best_model_path)
logger.info(f"Best model (Fold {best_fold_idx + 1}) saved to: {best_model_path}")

# Simpan CV fold details
cv_details_path = os.path.join(OUTPUT_DIR, "cv_fold_details.csv")
cv_details_df.to_csv(cv_details_path, index=False)
logger.info(f"CV fold details saved to: {cv_details_path}")

# Simpan CV metrics summary JSON
cv_summary_path = os.path.join(OUTPUT_DIR, "cv_metrics_summary.json")
with open(cv_summary_path, "w") as f:
    json.dump(cv_summary, f, indent=2)
logger.info(f"CV metrics summary saved to: {cv_summary_path}")

# Simpan Out-of-Fold (OOF) Predictions
oof_df = pd.DataFrame({
    "project_id": groups,
    "true_label": y,
    "oof_predicted_score": oof_predictions
})
oof_path = os.path.join(OUTPUT_DIR, "oof_predictions.csv")
oof_df.to_csv(oof_path, index=False)
logger.info(f"OOF predictions saved to: {oof_path}")

# Simpan daftar fitur
features_path = os.path.join(OUTPUT_DIR, "feature_list.txt")
with open(features_path, "w") as f:
    f.write("\n".join(X.columns.tolist()))
logger.info(f"Feature list saved to: {features_path}")

# Simpan dataset hasil pemrosesan lengkap
dataset_out = X.copy()
dataset_out["label"] = y
dataset_out["rank"] = df["rank"].values
dataset_out["project_id"] = groups
dataset_path = os.path.join(OUTPUT_DIR, "tfidf_puretext_dataset.csv")
dataset_out.to_csv(dataset_path, index=False)
logger.info(f"Processed dataset saved to: {dataset_path} (shape: {dataset_out.shape})")

Cell 25 executed successfully.


## STEP 12 — Pencatatan MLflow dan Ringkasan Akhir Eksperimen

Mencatat seluruh parameter Group K-Fold, metrik agregasi mean ± std (dengan sanitasi nama metrik agar kompatibel dengan server MLflow), dan artefak ke server MLflow tracking https://mlflow.smbgarasibmw.my.id/ serta menampilkan tabel ringkasan eksperimen.

In [14]:
if mlflow.active_run():
    mlflow.end_run()

try:
    with mlflow.start_run(run_name="TF-IDF PureText GKF Experiment Run") as run:
        # Log parameter eksperimen
        mlflow.log_param("text_representation", "PureText TF-IDF")
        mlflow.log_param("feature_selection_method", FEATURE_SELECTION_METHOD)
        mlflow.log_param("selected_k_features", len(selected_tfidf_cols))
        mlflow.log_param("total_fused_features", X.shape[1])
        mlflow.log_param("validation_strategy", "GroupKFold")
        mlflow.log_param("n_splits", N_SPLITS)
        mlflow.log_param("group_column", "project_id")
        mlflow.log_param("total_projects", len(np.unique(groups)))
        mlflow.log_param("ranker_objective", "lambdarank")
        mlflow.log_param("ranker_n_estimators", 100)
        mlflow.log_param("random_state", RANDOM_STATE)

        # Log metrik agregasi CV (mean dan std) — sanitasi '@' menjadi '_at_' agar valid di MLflow
        mlflow_metrics = {k.replace("@", "_at_"): float(v) for k, v in cv_summary.items()}
        mlflow.log_metrics(mlflow_metrics)

        # Log artefak
        mlflow.log_artifact(best_model_path, artifact_path="model")
        mlflow.log_artifact(cv_summary_path, artifact_path="metrics")
        mlflow.log_artifact(cv_details_path, artifact_path="metrics")
        mlflow.log_artifact(features_path, artifact_path="features")
        mlflow.log_artifact(dataset_path, artifact_path="dataset")
        mlflow.log_artifact(oof_path, artifact_path="predictions")

        logger.info(f"MLflow run ID: {run.info.run_id}")
        logger.info("MLflow experiment logged successfully with Group K-Fold metrics.")
except Exception as e:
    logger.warning(f"MLflow logging error: {e}. Experiment execution & local artifacts completed successfully.")

logger.info("=== Experiment Group K-Fold Completed Successfully ===")
display(cv_summary_df)

Cell 27 executed successfully.
